Produce a figure illustrating raw amplitude data with interference, interference removal and ice-ocean interface detection.

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import cmocean.cm as cmo
from matplotlib import rc

from adcp import find_interfered_pings, distance_to_interface

rc('font', size=7)
plt.rcParams['font.sans-serif'] = ['Arial'] + plt.rcParams['font.sans-serif'] # Arial as first choice

fig_width = 5.5 
fig_height = 3.5
cbar_aspect = 35

In [ ]:
ADCP_file = 'data/input/NBP2202_03_ADCP_echo_intensity.nc'
ds = xr.load_dataset(ADCP_file)
ds = ds.assign_coords(time_index=xr.DataArray((range(len(ds.time.values))), coords={'time':ds.time}))

In [ ]:
beam = 2
start = 8100
N =400
index = slice(start, start+N)
example_ping = 3216

In [ ]:
intensity_threshold = 140
percentage_threshold = 10

is_interfered = find_interfered_pings(ds, intensity_threshold = intensity_threshold, 
                                      percentage_threshold = percentage_threshold)
ds_cleaned = ds.where(~is_interfered)

In [ ]:
surface_range = distance_to_interface(ds_cleaned, interpolate=True, 
                                      rmin=200, rmax=1150, ampmin=100)
surface_range_vertical = surface_range * np.cos(np.radians(20)) # convert to vertical range

In [ ]:
def quadratic_interpolation_plot(ping, ax, amp_min, margin = 3.5, offset = 15, trianglesize=80):
    intensity = ds_cleaned.intensity.isel(time=ping).sel(beam=beam)
    surface_detection_cell = intensity.argmax(dim='range').values

    cell = np.arange(surface_detection_cell-1,surface_detection_cell+2)
    amp  = intensity.isel(range=slice(surface_detection_cell-1,surface_detection_cell+2))
    adjustment = - (amp[2]-amp[0])/(2.0*(amp[2]-2*amp[1]+amp[0]))

    # plot signal
    ax.plot(intensity, np.arange(ds_cleaned.range.size), '.-', c='gray',)

    # mark values used in the quadratic interpolation
    ax.scatter(amp,cell, c='k', s=trianglesize/2, zorder = 10)

    # plot interpolation
    param = np.polyfit(cell, amp, 2)
    cell_vals = np.linspace(cell[0]-2, cell[2]+2, 100)
    fit = param[2] + param[1]*cell_vals + param[0]*cell_vals**2
    fit[fit<amp_min]=np.nan
    ax.plot(fit, cell_vals, '--k')

    # Mark adjusted detection
    adjusted_detection = surface_detection_cell + adjustment
    adjusted_detection_amp = param[2] + param[1]*adjusted_detection + param[0]*adjusted_detection**2
    ax.scatter(adjusted_detection_amp + offset, adjusted_detection, c='r', s= trianglesize, marker='<', zorder=20)
    
    # zoom
    ax.set_ylim(surface_detection_cell - margin, surface_detection_cell + margin)
    ax.set_xlim(ax.get_xlim()+np.array([0,20]))

In [ ]:
plot_kwargs = {'x' : 'time_index',
               'vmin' : 50, 
               'vmax' : 160,
               'cmap' : cmo.tempo,
               'add_colorbar' : False,
              }

fig = plt.figure(figsize = (fig_width, fig_height/1.2))

gs0 = gridspec.GridSpec(1, 2, figure=fig, width_ratios=[5,1], wspace=0.03)

gs00 = gs0[0].subgridspec(2, 1, hspace = 0.05)

axa = fig.add_subplot(gs00[0,:])
axb = fig.add_subplot(gs00[1,:])

gs01 = gs0[1].subgridspec(1,1)

axc = fig.add_subplot(gs01[:, :])

# Panel 1: raw data
im = ds.intensity.isel(time=index).sel(beam=beam).plot(ax=axa,**plot_kwargs)
axa.set_xticks([])
axa.set_xlabel('')
axa.set_title('')

# Panel 2: detections
ds_cleaned.intensity.isel(time=index).sel(beam=beam).dropna(dim='time').plot(ax=axb, **plot_kwargs)
axb.scatter(ds_cleaned.isel(time=index).time_index, surface_range_vertical.isel(time=index).sel(beam=beam), c='w', s=4)
axb.set_xticks([])
axb.set_xlabel('Ping')
axb.set_title('')

# Panel 3: quadratic interpolation
quadratic_interpolation_plot(example_ping, axc, 65, offset = 15, trianglesize=30, margin = 5.5)
axc.yaxis.set_ticks_position('right')
axc.yaxis.set_label_position('right')
axc.set_ylabel('Cell')
axc.set_xlabel('Echo intensity')

# Neater axes
axes = [axa, axb]
ax_labels = ['(a)', '(b)']
label_colors = ['w', 'k']
for (ax, label, c) in zip(axes, ax_labels, label_colors):
    ax.set_ylabel('Range (m)')
    ax.annotate(label, xy=(0.01, 0.85), xycoords='axes fraction', color = c, weight='bold')
axc.annotate('(c)', xy=(0.09, 0.94), xycoords='axes fraction', color = 'k', weight='bold')

# Colorbar
cbar = fig.colorbar(im, ax=[axa,axb,axc], label = 'Echo intensity (count)', 
                    aspect = cbar_aspect, extend='both', orientation='horizontal', location='top', pad = 0.02)
cbar.ax.tick_params(length=2)

plt.savefig('figures/fig3.png', bbox_inches = 'tight', dpi=600)